# ดูตลาด (อ่านอย่างเดียว)

notebook นี้ **ใช้แค่ GET** (`/v1/version`, `/v1/calendar`, `/v1/state`) — ไม่ส่งซอง ไม่ reset ไม่กดข้าม · รันพร้อม agent ได้
key อ่านจาก `.env` (ของโฟลเดอร์นี้ หรือของ starter) ไม่ต้องพิมพ์ key ลงใน notebook

**วิธีใช้:** เลือก kernel = Python ของ starter `.venv` (มุมขวาบน) → รันช่อง 1 ครั้งเดียว → รันช่อง 2 (ดูครั้งเดียว) หรือช่อง 3 (ดูสดทุก 3 วินาที · หยุดด้วยปุ่ม ■)


In [ ]:
# 1) ตั้งค่า: อ่าน MARKET_URL / TEAM_KEY จาก .env (ไม่แสดง key)
import os, time
from pathlib import Path
import httpx
from IPython.display import clear_output

# ตลาดแข่ง: อ่าน .env.competition ก่อน (พอร์ต 8001) · ไม่มีไฟล์นี้ = ตลาดฝึกจาก .env ของ starter
# ค่าที่อ่านเก็บแยก (ไม่ใช้ค่าค้างใน kernel) → รันช่องนี้ใหม่ได้เลย ไม่ต้องรีสตาร์ท kernel
cfg = {}
for env in (Path(".env.competition"), Path(".env"), Path(r"C:\Users\guttj\Documents\TGR2026\starter\Coffee-TESA-starter\.env")):
    if env.exists():
        for line in env.read_text(encoding="utf-8").splitlines():
            if "=" in line and not line.lstrip().startswith("#"):
                k, v = line.split("=", 1)
                if v.strip():
                    cfg.setdefault(k.strip(), v.strip())

api = httpx.Client(base_url=cfg["MARKET_URL"].rstrip("/"), timeout=10,
                   headers={"Authorization": f"Bearer {cfg['TEAM_KEY']}", "User-Agent": "modengi-viewer/1.0"})
mode = api.get("/v1/version").json().get("mode")
rules = api.get("/v1/calendar").json()["rules"]
print("ตลาด:", cfg["MARKET_URL"], "· mode:", mode, "· รอบ:", rules["timeline"]["rounds"])

In [2]:
# 2) ภาพตลาดตอนนี้ (รันซ้ำได้เรื่อยๆ)
def show():
    s = api.get("/v1/state").json()
    rnd = s.get("round") or {}
    print(f"ช่วง {s['phase']} · รอบ {rnd.get('round_id', '-')} ({rnd.get('calendar', '')}) · เหลือ {round(s.get('seconds_left') or 0)} วิ"
          f" · capacity เหลือ {s.get('capacity_left_kg')} กก. · ประเมินแล้ว {s.get('appraisal_submitted')}")
    st = s.get("standings") or {}
    print(f"ขายแล้ว {st.get('kg_sold', 0)} กก. · รายได้ {st.get('revenue_thb', 0):,.0f} บาท · เหลือ {st.get('kg_remaining')} / {st.get('kg_total')} กก.")
    print("\nล็อต:")
    for l in s["lots"]:
        sm = l["sample"]
        print(f"  {l['lot_id']:<11} ว่าง {l['kg_available']:>4} กก. (เปิดซองอยู่ {l['kg_in_open_offers']}) · defect {sm['defect_pct']} ชื้น {sm['moisture_pct']} screen {sm['screen']}")
    if s["rfqs"]:
        print("\nRFQ รอบนี้:")
        for r in s["rfqs"]:
            sp = r["spec"]
            print(f"  {r['rfq_id']} {r['buyer_type']:<10} {r['buyer']:<18} {r['bags']:>3} กระสอบ · สเปก defect≤{sp['max_defect_pct']} ชื้น≤{sp['max_moisture_pct']} screen≥{sp['min_screen']}")
    if s["my_open_offers"]:
        print("\nซองที่ agent เปิดอยู่:")
        for o in s["my_open_offers"]:
            print(f"  {o['rfq_id']} {o['lot_id']} {o['kg']} กก. @ {o['price_per_kg']}")
    lr = s.get("last_results") or {}
    if lr.get("round_id"):
        print(f"\nผลรอบ {lr['round_id']} (ของเรา):")
        for o in lr["my_offers"]:
            print(f"  {o['rfq_id']} {o['lot_id']} {o['kg']} กก. @ {o['price_per_kg']} → {o['status']} ({o.get('filled_kg', 0)} กก.)")
        print("ภาพรวมตลาดรอบนั้น:")
        for m in lr["market_summary"]:
            fill = m['bought_kg'] / max(m['wanted_kg'], 1)
            print(f"  {m['rfq_id']} {m['buyer_type']:<10} ซื้อได้ {fill:>4.0%} · {m['offers']:>3} ข้อเสนอ · ราคา {m.get('price_min')}–{m.get('price_max')} (เฉลี่ย {m.get('price_avg')})")

show()

ช่วง ROUND_OPEN · รอบ R3 (ปกติ) · เหลือ 179 วิ · capacity เหลือ None กก. · ประเมินแล้ว True
ขายแล้ว 360 กก. · รายได้ 43,683 บาท · เหลือ 1200 / 1560 กก.

ล็อต:
  LOT-T13-1   ว่าง  360 กก. (เปิดซองอยู่ 120) · defect 4.87 ชื้น 11.9 screen 16
  LOT-T13-2   ว่าง   60 กก. (เปิดซองอยู่ 120) · defect 3.51 ชื้น 12.42 screen 16
  LOT-T13-3   ว่าง  240 กก. (เปิดซองอยู่ 120) · defect 1.63 ชื้น 10.93 screen 14
  LOT-T13-4   ว่าง  180 กก. (เปิดซองอยู่ 0) · defect 11.79 ชื้น 14.18 screen 15

RFQ รอบนี้:
  R3-01 INSTANT    5 อินสแตนท์ไทย      10 กระสอบ · สเปก defect≤15.0 ชื้น≤14.0 screen≥None
  R3-02 COMMERCIAL 3 สยามคอมเมิร์ซ     11 กระสอบ · สเปก defect≤7.0 ชื้น≤12.5 screen≥None
  R3-03 INSTANT    6 ทรีอินวันฟู้ดส์    6 กระสอบ · สเปก defect≤15.0 ชื้น≤14.0 screen≥None
  R3-04 INSTANT    5 อินสแตนท์ไทย       7 กระสอบ · สเปก defect≤15.0 ชื้น≤14.0 screen≥None
  R3-05 COMMERCIAL 4 ชุมพรคั่วบด       13 กระสอบ · สเปก defect≤7.0 ชื้น≤12.5 screen≥None
  R3-06 COMMERCIAL 4 ชุมพรคั่วบด        8 กระสอบ · สเปก de

In [ ]:
# 3) ดูสด ทุก 3 วินาที (หยุดด้วยปุ่ม ■ Interrupt) · อ่านอย่างเดียว ~0.3 ครั้ง/วินาที
try:
    while True:
        clear_output(wait=True)
        show()
        time.sleep(3)
except KeyboardInterrupt:
    print("หยุดดูแล้ว")

ช่วง ROUND_OPEN · รอบ R2 (ปกติ) · เหลือ 180 วิ · capacity เหลือ None กก. · ประเมินแล้ว True
ขายแล้ว 0 กก. · รายได้ 0 บาท · เหลือ 1620 / 1620 กก.

ล็อต:
  LOT-T13-1   ว่าง  600 กก. (เปิดซองอยู่ 0) · defect 2.16 ชื้น 11.21 screen 18
  LOT-T13-2   ว่าง  420 กก. (เปิดซองอยู่ 0) · defect 3.87 ชื้น 12.19 screen 15
  LOT-T13-3   ว่าง   60 กก. (เปิดซองอยู่ 0) · defect 8.0 ชื้น 14.33 screen 13
  LOT-T13-4   ว่าง  540 กก. (เปิดซองอยู่ 0) · defect 5.16 ชื้น 11.64 screen 16

RFQ รอบนี้:
  R2-01 COMMERCIAL 3 สยามคอมเมิร์ซ     11 กระสอบ · สเปก defect≤7.0 ชื้น≤12.5 screen≥None
  R2-02 INSTANT    6 ทรีอินวันฟู้ดส์    8 กระสอบ · สเปก defect≤15.0 ชื้น≤14.0 screen≥None
  R2-03 INSTANT    5 อินสแตนท์ไทย       9 กระสอบ · สเปก defect≤15.0 ชื้น≤14.0 screen≥None
  R2-04 COMMERCIAL 4 ชุมพรคั่วบด        9 กระสอบ · สเปก defect≤7.0 ชื้น≤12.5 screen≥None
  R2-05 INSTANT    6 ทรีอินวันฟู้ดส์    8 กระสอบ · สเปก defect≤15.0 ชื้น≤14.0 screen≥None
  R2-06 INSTANT    5 อินสแตนท์ไทย       9 กระสอบ · สเปก defect≤15.0 ชื้น